In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_3.py ──
"""
# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 3: Shared Utilities
# ════════════════════════════════════════════════════════════════════════
#
# Common data loading, windowing, training, visualisation, and experiment
# tracking utilities shared across all technique files in this exercise.
#
# This module is NOT meant to be run standalone — import it from the
# technique files (01_vanilla_rnn.py, 02_lstm.py, etc.).
# ════════════════════════════════════════════════════════════════════════
"""

import asyncio
import pickle
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec

# ── Constants ───────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "stocks"
OUTPUT_DIR = REPO_ROOT / "outputs" / "mlfp05" / "ex_3"

TICKERS = {
    "^STI": "Straits Times Index",
    "D05.SI": "DBS Group Holdings",
    "9988.HK": "Alibaba HK",
    "AAPL": "Apple",
    "005930.KS": "Samsung",
    "7203.T": "Toyota",
}

SEQ_LEN = 20  # 20-day lookback (4 trading weeks)
FORECAST_HORIZON = 5  # predict next 5 days
FEATURES = ["Close", "High", "Low", "Volume"]
# Indicator-augmented feature set for 06_technical_indicators.py. Every
# indicator is computed from PAST closes only (rolling/ewm windows), so no
# future information leaks into the features.
INDICATOR_FEATURES = FEATURES + ["RSI", "MACD_hist", "BB_pctB"]
HIDDEN_DIM = 64
EPOCHS = 15
LR = 1e-3
CLIP = 1.0
BATCH_SIZE = 64


def init_environment() -> torch.device:
    """Set up environment, seeds, device, and output directories."""
    setup_environment()
    torch.manual_seed(42)
    np.random.seed(42)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    device = get_device()
    print(f"Using device: {device}")
    return device


# ── Data Loading ────────────────────────────────────────────────────────
def fetch_ticker(symbol: str) -> pl.DataFrame:
    """Download daily OHLCV bars from yfinance, return polars DataFrame."""
    import yfinance as yf

    df = yf.download(
        symbol, start="2010-01-01", end="2024-12-31", progress=False, auto_adjust=True
    )
    if df is None or len(df) == 0:
        raise RuntimeError(f"yfinance returned empty frame for {symbol}")
    df = df.copy()
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    return pl.from_pandas(df.reset_index())


def load_or_fetch(symbol: str) -> tuple[pl.DataFrame | None, str]:
    """Load from parquet cache, or download and cache."""
    cache = DATA_DIR / f"{symbol.replace('^', '').replace('.', '_')}.parquet"
    if cache.exists():
        return pl.read_parquet(cache), "cache"
    try:
        df = fetch_ticker(symbol)
        df.write_parquet(cache)
        return df, "yfinance"
    except Exception as exc:
        print(f"  {symbol} unavailable ({type(exc).__name__}: {exc})")
        return None, "failed"


def load_stock_data() -> tuple[dict[str, pl.DataFrame], str, pl.DataFrame]:
    """Load all tickers and return (stock_data, primary_symbol, primary_df)."""
    stock_data: dict[str, pl.DataFrame] = {}
    for symbol, name in TICKERS.items():
        df, source = load_or_fetch(symbol)
        if df is not None:
            stock_data[symbol] = df
            print(f"  {symbol} ({name}): {len(df)} days [{source}]")

    if "^STI" not in stock_data and "AAPL" not in stock_data:
        raise RuntimeError("Need at least ^STI or AAPL data to proceed")

    primary = "^STI" if "^STI" in stock_data else "AAPL"
    primary_df = stock_data[primary]
    print(
        f"\nPrimary: {primary} -- {len(primary_df)} days, "
        f"{primary_df['Date'].min()} -> {primary_df['Date'].max()}"
    )
    return stock_data, primary, primary_df


# ── Technical Indicators (polars-native) ─────────────────────────────
def add_technical_indicators(df: pl.DataFrame) -> pl.DataFrame:
    """Append RSI(14), MACD(12,26,9) and Bollinger %B(20,2) in polars.

    All three are the canonical definitions:
      RSI:   Wilder's smoothed gain/loss ratio, 100 - 100/(1 + RS)
      MACD:  EMA(12) - EMA(26); signal = EMA(9) of MACD; hist = MACD - signal
      %B:    (Close - lower) / (upper - lower) with 20-day mean +/- 2 sd

    Every expression uses backward-looking rolling/ewm windows only, so the
    indicator on day t is computable at the close of day t — no lookahead.
    The first ~26 rows carry nulls (window warm-up); callers should
    ``drop_nulls()`` before building windowed datasets.
    """
    return df.with_columns(
        # ── RSI(14), Wilder's smoothing (ewm alpha = 1/14) ──────────
        pl.col("Close").diff().alias("_delta"),
    ).with_columns(
        pl.when(pl.col("_delta") > 0)
        .then(pl.col("_delta"))
        .otherwise(0.0)
        .alias("_gain"),
        pl.when(pl.col("_delta") < 0)
        .then(-pl.col("_delta"))
        .otherwise(0.0)
        .alias("_loss"),
    ).with_columns(
        pl.col("_gain").ewm_mean(alpha=1 / 14, adjust=False).alias("_avg_gain"),
        pl.col("_loss").ewm_mean(alpha=1 / 14, adjust=False).alias("_avg_loss"),
    ).with_columns(
        (
            100.0
            - 100.0 / (1.0 + pl.col("_avg_gain") / (pl.col("_avg_loss") + 1e-12))
        ).alias("RSI"),
        # ── MACD(12, 26, 9) ─────────────────────────────────────────
        (
            pl.col("Close").ewm_mean(span=12, adjust=False)
            - pl.col("Close").ewm_mean(span=26, adjust=False)
        ).alias("_macd"),
    ).with_columns(
        pl.col("_macd").ewm_mean(span=9, adjust=False).alias("_macd_signal"),
        # ── Bollinger %B(20, 2) ─────────────────────────────────────
        pl.col("Close").rolling_mean(20).alias("_bb_mid"),
        pl.col("Close").rolling_std(20).alias("_bb_std"),
    ).with_columns(
        (pl.col("_macd") - pl.col("_macd_signal")).alias("MACD_hist"),
        (
            (
                pl.col("Close")
                - (pl.col("_bb_mid") - 2.0 * pl.col("_bb_std"))
            )
            / (4.0 * pl.col("_bb_std") + 1e-12)
        ).alias("BB_pctB"),
    ).drop("_delta", "_gain", "_loss", "_avg_gain", "_avg_loss",
           "_macd", "_macd_signal", "_bb_mid", "_bb_std")


# ── Text Corpus (character-level modelling) ───────────────────────────
TEXT_DIR = REPO_ROOT / "data" / "mlfp05" / "text"
TINY_SHAKESPEARE_URL = (
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/"
    "tinyshakespeare/input.txt"
)


def load_text_corpus() -> tuple[str, str]:
    """Load a character-level text corpus, cached locally.

    Primary source: tiny-shakespeare (~1.1 MB of Shakespeare dialogue, the
    canonical char-RNN corpus), downloaded once and cached under
    ``data/mlfp05/text/``. If the download is unavailable (offline lab),
    the module's own textbook is used — the model then learns to write
    course-style prose, which is itself a fun demonstration.

    Returns: (text, corpus_name)
    """
    TEXT_DIR.mkdir(parents=True, exist_ok=True)
    cache = TEXT_DIR / "tiny_shakespeare.txt"
    if cache.exists():
        return cache.read_text(encoding="utf-8"), "tiny-shakespeare (cached)"

    import urllib.request

    try:
        with urllib.request.urlopen(TINY_SHAKESPEARE_URL, timeout=30) as resp:
            text = resp.read().decode("utf-8")
        cache.write_text(text, encoding="utf-8")
        return text, "tiny-shakespeare (downloaded)"
    except Exception as exc:
        textbook = REPO_ROOT / "modules" / "mlfp05" / "textbook.md"
        if textbook.exists():
            print(
                f"  tiny-shakespeare unavailable ({type(exc).__name__}); "
                "using the module textbook as corpus"
            )
            return textbook.read_text(encoding="utf-8"), "mlfp05 textbook"
        raise RuntimeError(
            f"No text corpus available: download failed ({exc}) and no "
            f"local textbook at {textbook}"
        ) from exc


# ── Windowed Datasets ───────────────────────────────────────────────────
def build_dataset(
    df: pl.DataFrame,
    seq_len: int = SEQ_LEN,
    horizon: int = FORECAST_HORIZON,
    feature_cols: list[str] | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, int]:
    """Build (seq_len window) -> (next horizon closes) arrays with z-score normalisation.

    Args:
        df: OHLCV frame (optionally indicator-augmented)
        seq_len: lookback window length
        horizon: forecast horizon
        feature_cols: columns to use as features (default: FEATURES).
            The TARGET is always the first column of feature_cols and must
            be "Close" — normalisation stats for the target come from the
            train split only.

    Returns: X, y, mean, std, n_train_windows
    """
    cols = FEATURES if feature_cols is None else feature_cols
    if cols[0] != "Close":
        raise ValueError(
            f"feature_cols[0] must be 'Close' (the forecast target); got {cols[0]!r}"
        )
    data = df.select(cols).to_numpy().astype(np.float32)
    n = len(data)
    split_n = int(0.8 * n)
    train_data = data[:split_n]
    mean = train_data.mean(axis=0, keepdims=True)
    std = train_data.std(axis=0, keepdims=True) + 1e-8
    data_norm = (data - mean) / std

    n_windows = n - seq_len - horizon + 1
    X = np.stack([data_norm[i : i + seq_len] for i in range(n_windows)])
    y = np.stack(
        [data_norm[i + seq_len : i + seq_len + horizon, 0] for i in range(n_windows)]
    )
    split_idx = split_n - seq_len
    return X.astype(np.float32), y.astype(np.float32), mean, std, split_idx


def prepare_dataloaders(
    primary_df: pl.DataFrame,
    device: torch.device,
    feature_cols: list[str] | None = None,
) -> tuple[
    DataLoader,
    DataLoader,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    np.ndarray,
    np.ndarray,
    int,
    int,
]:
    """Build train/val dataloaders and return raw tensors plus normalisation stats.

    Returns: train_loader, val_loader, X_train_t, y_train_t, X_val_t, y_val_t,
             norm_mean, norm_std, n_train_w, n_features
    """
    X_all, y_all, norm_mean, norm_std, n_train_w = build_dataset(
        primary_df, feature_cols=feature_cols
    )
    print(
        f"Built {len(X_all)} windows (seq_len={SEQ_LEN}, horizon={FORECAST_HORIZON}); "
        f"train {n_train_w}, val {len(X_all) - n_train_w}"
    )

    X_train_t = torch.from_numpy(X_all[:n_train_w]).to(device)
    y_train_t = torch.from_numpy(y_all[:n_train_w]).to(device)
    X_val_t = torch.from_numpy(X_all[n_train_w:]).to(device)
    y_val_t = torch.from_numpy(y_all[n_train_w:]).to(device)
    print(f"  X_train: {tuple(X_train_t.shape)}  y_train: {tuple(y_train_t.shape)}")
    print(f"  X_val:   {tuple(X_val_t.shape)}    y_val:   {tuple(y_val_t.shape)}")

    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True
    )
    val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE)
    n_features = X_train_t.shape[-1]

    return (
        train_loader,
        val_loader,
        X_train_t,
        y_train_t,
        X_val_t,
        y_val_t,
        norm_mean,
        norm_std,
        n_train_w,
        n_features,
    )


# ── Experiment Tracking ─────────────────────────────────────────────────
async def _setup_engines(
    primary: str,
    experiment_suffix: str = "",
) -> tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None, bool]:
    """Create ExperimentTracker (kailash-ml 1.1.1 factory) and ModelRegistry."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_rnns.db"
    registry_db = "sqlite:///mlfp05_rnns_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    exp_name = (
        f"m5_rnns_{primary}_{experiment_suffix}"
        if experiment_suffix
        else f"m5_rnns_{primary}"
    )
    return conn, tracker, exp_name, registry, True


def setup_engines(
    primary: str,
    experiment_suffix: str = "",
) -> tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None, bool]:
    """Sync wrapper for engine setup."""
    return asyncio.run(_setup_engines(primary, experiment_suffix))


# ── Training Harness ────────────────────────────────────────────────────
def compute_gradient_norm(model: nn.Module) -> float:
    """Compute the total L2 norm of all gradients (before clipping)."""
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item() ** 2
    return total_norm**0.5


def _predict(model: nn.Module, x: torch.Tensor, attn: bool = False) -> torch.Tensor:
    """Forward pass, handling attention models that return a tuple."""
    out = model(x)
    return out[0] if attn else out


async def _train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    epochs: int = EPOCHS,
    lr: float = LR,
    clip: float = CLIP,
    attn: bool = False,
) -> dict[str, Any]:
    """Train with gradient tracking, log to ExperimentTracker."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses, val_losses, gradient_norms = [], [], []
    n_params = sum(p.numel() for p in model.parameters())

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "hidden_dim": str(HIDDEN_DIM),
                "seq_len": str(SEQ_LEN),
                "forecast_horizon": str(FORECAST_HORIZON),
                "epochs": str(epochs),
                "lr": str(lr),
                "clip_norm": str(clip),
                "n_params": str(n_params),
            }
        )
        print(f"  [{name}] {n_params:,} parameters")

        for epoch in range(epochs):
            model.train()
            b_losses, e_grads = [], []
            for xb, yb in train_loader:
                opt.zero_grad()
                loss = F.mse_loss(_predict(model, xb, attn), yb)
                loss.backward()
                e_grads.append(compute_gradient_norm(model))
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
                opt.step()
                b_losses.append(loss.item())

            tl, gn = float(np.mean(b_losses)), float(np.mean(e_grads))
            train_losses.append(tl)
            gradient_norms.append(gn)

            model.eval()
            with torch.no_grad():
                vl = float(
                    np.mean(
                        [
                            F.mse_loss(_predict(model, xb, attn), yb).item()
                            for xb, yb in val_loader
                        ]
                    )
                )
            val_losses.append(vl)

            await run.log_metrics(
                {"train_loss": tl, "val_loss": vl, "gradient_norm": gn},
                step=epoch + 1,
            )
            print(
                f"  [{name}] epoch {epoch+1:2d}/{epochs}  "
                f"train={tl:.4f}  val={vl:.4f}  grad={gn:.4f}"
            )

        await run.log_metric("final_val_loss", val_losses[-1])

    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "gradient_norms": gradient_norms,
        "final_val_loss": val_losses[-1],
    }


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    epochs: int = EPOCHS,
    lr: float = LR,
    clip: float = CLIP,
    attn: bool = False,
) -> dict[str, Any]:
    """Sync wrapper for training."""
    return asyncio.run(
        _train_model_async(
            model,
            name,
            tracker,
            exp_name,
            train_loader,
            val_loader,
            device,
            epochs,
            lr,
            clip,
            attn,
        )
    )


# ── Model Registry ──────────────────────────────────────────────────────
def register_best_model(
    model: nn.Module,
    model_name: str,
    val_loss: float,
    primary: str,
    registry: ModelRegistry | None,
    has_registry: bool,
) -> None:
    """Register a model in the ModelRegistry."""
    if not has_registry or registry is None:
        print("  ModelRegistry not available, skipping registration")
        return

    model_bytes = pickle.dumps(model.state_dict())
    # Architecture/ticker are encoded in the model name; numeric facts go to MetricSpec.
    reg_result = asyncio.run(
        registry.register_model(
            name=f"m5_rnn_{model_name.lower()}_{primary.replace('^', '')}",
            artifact=model_bytes,
            metrics=[
                MetricSpec(
                    name="val_loss", value=float(val_loss), higher_is_better=False
                ),
                MetricSpec(name="hidden_dim", value=float(HIDDEN_DIM)),
                MetricSpec(name="seq_len", value=float(SEQ_LEN)),
                MetricSpec(name="forecast_horizon", value=float(FORECAST_HORIZON)),
                MetricSpec(name="epochs", value=float(EPOCHS)),
            ],
        )
    )
    print(f"  Registered: {reg_result.name} v{reg_result.version}")


# ── Visualisation Helpers ───────────────────────────────────────────────
def get_visualizer() -> ModelVisualizer:
    """Create a ModelVisualizer instance.

    ModelVisualizer carries a kailash-ml P2 experimental notice (a
    UserWarning at construction); exercises run under warnings-as-errors,
    so the notice is acknowledged narrowly at this construction site.
    """
    import warnings as _warnings

    from kailash_ml._decorators import ExperimentalWarning as _EW

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", _EW)
        return ModelVisualizer()


def plot_training_curves(
    viz: ModelVisualizer,
    results: dict[str, Any],
    model_name: str,
    output_prefix: str,
) -> None:
    """Plot training/validation loss curves and gradient norms."""
    train_metrics = {
        f"{model_name} train": results["train_losses"],
        f"{model_name} val": results["val_losses"],
    }
    viz.training_history(
        metrics=train_metrics, x_label="Epoch", y_label="MSE Loss"
    ).write_html(str(OUTPUT_DIR / f"{output_prefix}_training_curves.html"))

    grad_metrics = {model_name: results["gradient_norms"]}
    viz.training_history(
        metrics=grad_metrics, x_label="Epoch", y_label="Gradient L2 Norm"
    ).write_html(str(OUTPUT_DIR / f"{output_prefix}_gradient_norms.html"))


def plot_predictions(
    viz: ModelVisualizer,
    model: nn.Module,
    X_val_t: torch.Tensor,
    y_val_t: torch.Tensor,
    norm_mean: np.ndarray,
    norm_std: np.ndarray,
    output_prefix: str,
    attn: bool = False,
) -> tuple[np.ndarray, np.ndarray, torch.Tensor | None]:
    """Generate prediction-vs-actual scatter and return denormalised arrays.

    Returns: preds_denorm, actual_denorm, attn_weights (or None)
    """
    model.eval()
    with torch.no_grad():
        if attn:
            val_preds, attn_weights = model(X_val_t)
        else:
            val_preds = model(X_val_t)
            attn_weights = None

    close_mean, close_std = norm_mean[0, 0], norm_std[0, 0]
    preds_denorm = val_preds.cpu().numpy() * close_std + close_mean
    actual_denorm = y_val_t.cpu().numpy() * close_std + close_mean

    pred_df = pl.DataFrame(
        {
            "actual": actual_denorm[:, 0].tolist(),
            "predicted": preds_denorm[:, 0].tolist(),
        }
    )
    viz.scatter(pred_df, x="actual", y="predicted").write_html(
        str(OUTPUT_DIR / f"{output_prefix}_pred_vs_actual.html")
    )

    return preds_denorm, actual_denorm, attn_weights


def plot_time_series_overlay(
    preds_denorm: np.ndarray,
    actual_denorm: np.ndarray,
    output_prefix: str,
    title: str = "Predicted vs Actual Close Price",
    n_points: int = 200,
) -> None:
    """Plot predicted vs actual as overlaid time-series lines."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    n = min(n_points, len(preds_denorm))
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(
        range(n), actual_denorm[:n, 0], label="Actual", color="#2196F3", linewidth=1.5
    )
    ax.plot(
        range(n),
        preds_denorm[:n, 0],
        label="Predicted",
        color="#FF5722",
        linewidth=1.5,
        linestyle="--",
        alpha=0.85,
    )
    ax.set_xlabel("Validation Window Index")
    ax.set_ylabel("Close Price")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(str(OUTPUT_DIR / f"{output_prefix}_time_series_overlay.png"), dpi=150)
    plt.close(fig)
    print(f"  Saved: {output_prefix}_time_series_overlay.png")


def plot_horizon_error(
    preds_denorm: np.ndarray,
    actual_denorm: np.ndarray,
    model_name: str,
) -> list[float]:
    """Print and return RMSE by forecast horizon day."""
    print(f"\n  Forecast Error by Horizon Day ({model_name}):")
    rmses = []
    for day in range(FORECAST_HORIZON):
        rmse = (
            float(np.mean((preds_denorm[:, day] - actual_denorm[:, day]) ** 2)) ** 0.5
        )
        rmses.append(rmse)
        print(f"    Day {day + 1}: RMSE={rmse:.2f}")
    return rmses




# ════════════════════════════════════════════════════════════════════════
# MLFP05 Exercise 3.7 — Character-Level Text Generation with an LSTM:
# Perplexity as the Honest Metric
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this file, you will be able to:
#   - Frame text generation as next-character CLASSIFICATION over a
#     character vocabulary (the same cross-entropy as any classifier)
#   - Build the shifted-window dataset: input chars[i:i+L], target
#     chars[i+1:i+L+1]
#   - Train a 2-layer character LSTM and report PERPLEXITY =
#     exp(mean cross-entropy) — "the model's effective branching factor"
#   - Sample with temperature and explain the creativity dial honestly
#   - Compare a trained model's perplexity against the uniform-random
#     baseline (vocab size) — never report perplexity without it
#   - Apply to boilerplate drafting assistance at a professional
#     services firm
#
# PREREQUISITES: M5/ex_3/02_lstm.py (LSTM mechanics); 06 for feature
#   engineering mindset. No stock data here — the corpus is text.
# ESTIMATED TIME: ~35 min
#
# DATASET: tiny-shakespeare (~1.1 MB of Shakespeare dialogue, the
#   canonical char-RNN corpus), downloaded once and cached; falls back
#   to this module's own textbook if the lab is offline.
#
# PHASES:
#   1. THEORY  — next-char classification; perplexity; temperature
#   2. BUILD   — vocabulary, encode/decode, CharLSTM
#   3. TRAIN   — 2-layer LSTM, tracked with ExperimentTracker
#   4. VISUALISE — CE/perplexity curves + temperature samples
#   5. APPLY   — drafting assistance; what perplexity means to a partner
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


device = init_environment()



## PHASE 1 — THEORY: Generation = Classification, Repeated


GENERATION = CLASSIFICATION, REPEATED
    window of L characters -> softmax over vocabulary -> sample -> slide

  PERPLEXITY = exp(mean cross-entropy)
    "as uncertain as a uniform choice over P characters"
    uniform baseline over V characters: CE = ln(V), perplexity = V
    trained model: single digits. Report BOTH or report neither.

  TEMPERATURE: logits / T before softmax
    T=0.5 conservative | T=1.0 as-learned | T=1.5 adventurous


In [ ]:
# A language model is a classifier whose classes are CHARACTERS. Given
# the last L characters, predict the distribution of the next one.
# Sample from that distribution, append, slide the window, repeat.
#
# WHY CHARACTERS (not words)?
#   No tokeniser, no out-of-vocabulary problem, and the model must learn
#   spelling, punctuation, spaces — even CAPITALISATION conventions —
#   from scratch. That makes the learning VISIBLE: epoch-1 output is
#   gibberish; epoch-3 output has real words and dialogue structure.
#
# PERPLEXITY = exp(mean cross-entropy):
#   Cross-entropy is the average surprisal in nats. Its exponential has
#   a beautiful reading: "on average, the model is as uncertain as a
#   uniform choice over perplexity-many characters".
#     - Uniform random over a 70-char vocabulary: CE = ln(70) = 4.25,
#       perplexity = 70. This is the floor any trained model must beat.
#     - A trained char-LSTM on this corpus: perplexity in the single
#       digits — the model has narrowed 70 candidates to a handful.
#   NEVER report perplexity without the vocabulary-size reference.
#
# TEMPERATURE — the creativity dial:
#   Sampling divides logits by T before the softmax.
#     T < 1: sharpen — the model plays safe (repetitive, grammatical)
#     T = 1: sample the learned distribution as-is
#     T > 1: flatten — more surprise, more errors
#   There is no "best" T; there is a best T FOR THE TASK (see Phase 5).

print("=" * 70)
print("  PHASE 1 — THEORY: next-char classification and perplexity")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: vocabulary, encode/decode, CharLSTM


Characters -> vocabulary indices.


Vocabulary indices -> characters.


Output shape (batch, seq, vocab): EVERY timestep predicts the next
    character, so one forward pass yields seq_len classification losses.


Random windows: input chars[i:i+L], target chars[i+1:i+L+1].


Mean cross-entropy over fixed windows of a held-out sequence.


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: character vocabulary + CharLSTM")
print("=" * 70)

text, corpus_name = load_text_corpus()
print(f"Corpus: {corpus_name} — {len(text):,} characters")

chars = sorted(set(text))
VOCAB_SIZE = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}


def encode(s: str) -> list[int]:
    # TODO: map each character through char_to_idx
    return ____


def decode(indices) -> str:
    # TODO: map each index through idx_to_char and join into a string
    # Hint: "".join(...)
    return ____


# Contiguous split: first 90% train, last 10% validation. Text has no
# i.i.d. assumption to honour, but the val text must be CONTIGUOUS and
# NEVER seen in training — shuffling characters would be nonsense.
split_at = int(0.9 * len(text))
train_ids = np.array(encode(text[:split_at]), dtype=np.int64)
val_ids = np.array(encode(text[split_at:]), dtype=np.int64)
print(
    f"Vocabulary: {VOCAB_SIZE} characters | train {len(train_ids):,} chars, "
    f"val {len(val_ids):,} chars (contiguous split)"
)

SEQ_LEN_CHARS = 128
EMB_DIM = 64
HIDDEN_CHARS = 256
N_LAYERS = 2
CHAR_BATCH = 64
CHAR_EPOCHS = 3
STEPS_PER_EPOCH = 400
CHAR_LR = 2e-3


class CharLSTM(nn.Module):

    def __init__(self, vocab_size: int = VOCAB_SIZE, emb_dim: int = EMB_DIM,
                 hidden: int = HIDDEN_CHARS, n_layers: int = N_LAYERS):
        super().__init__()
        # TODO: embedding (vocab -> emb_dim), a 2-layer batch-first LSTM
        #   (emb_dim -> hidden), and a linear head (hidden -> vocab)
        self.emb = ____
        self.lstm = ____
        self.head = ____

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: embed, run the LSTM, map every timestep to vocab logits
        #   (keep the sequence dimension: output (batch, seq, vocab))
        out, _ = ____
        return ____


def sample_batch(ids: np.ndarray, seq_len: int = SEQ_LEN_CHARS,
                 batch_size: int = CHAR_BATCH) -> tuple[torch.Tensor, torch.Tensor]:
    # TODO: draw batch_size random start indices, each leaving room for
    #   seq_len + 1 characters (window PLUS the shifted target)
    starts = ____
    x = np.stack([ids[s : s + seq_len] for s in starts])
    # TODO: the target window — the SAME window shifted right by one char
    y = ____
    return (
        torch.from_numpy(x).to(device),
        torch.from_numpy(y).to(device),
    )


@torch.no_grad()
def mean_ce(ids: np.ndarray, model: nn.Module, n_batches: int = 25) -> float:
    model.eval()
    losses = []
    for k in range(n_batches):
        # evenly spaced windows (deterministic, covers the val text)
        starts = np.linspace(0, len(ids) - SEQ_LEN_CHARS - 1,
                             num=CHAR_BATCH, dtype=np.int64)
        x = torch.from_numpy(
            np.stack([ids[s : s + SEQ_LEN_CHARS] for s in starts])
        ).to(device)
        y = torch.from_numpy(
            np.stack([ids[s + 1 : s + SEQ_LEN_CHARS + 1] for s in starts])
        ).to(device)
        logits = model(x)
        losses.append(
            F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE), y.reshape(-1)
            ).item()
        )
    return float(np.mean(losses))



### Checkpoint 1: encode/decode round-trip and model shapes


In [ ]:
probe = text[:1000]
assert decode(encode(probe)) == probe, "encode/decode must round-trip"
_model_probe = CharLSTM().to(device)
_x, _y = sample_batch(train_ids)
_logits = _model_probe(_x)
assert _logits.shape == (CHAR_BATCH, SEQ_LEN_CHARS, VOCAB_SIZE), (
    f"CharLSTM logits should be (batch, seq, vocab) = "
    f"({CHAR_BATCH}, {SEQ_LEN_CHARS}, {VOCAB_SIZE}), got {_logits.shape}"
)
_n_params = sum(p.numel() for p in _model_probe.parameters())
uniform_ce = float(np.log(VOCAB_SIZE))
print(f"\nCharLSTM built: {_n_params:,} parameters")
print(f"  logits shape verified: {tuple(_logits.shape)}")
print(f"  uniform-random baseline: CE = ln({VOCAB_SIZE}) = {uniform_ce:.3f}, "
      f"perplexity = {VOCAB_SIZE}")
print(f"  encode/decode round-trip: OK ({len(probe)} chars)")
print("\n--- Checkpoint 1 passed --- vocabulary and model verified\n")
del _model_probe, _x, _y, _logits



## PHASE 3 — TRAIN: 2-layer char LSTM, tracked with ExperimentTracker


Train and log per-epoch CE/perplexity to ExperimentTracker.


In [ ]:
print("=" * 70)
print("  PHASE 3 — TRAIN: char LSTM on next-character prediction")
print("=" * 70)

conn, tracker, exp_name, registry, has_registry = setup_engines(
    "text", experiment_suffix="charlstm"
)

torch.manual_seed(42)
np.random.seed(42)
model = CharLSTM().to(device)
opt = torch.optim.Adam(model.parameters(), lr=CHAR_LR)

train_ces: list[float] = []
val_ces: list[float] = []
val_ppls: list[float] = []


async def train_char_lstm() -> None:
    async with tracker.track(experiment=exp_name, run_name="char_lstm") as run:
        await run.log_params(
            {
                "architecture": "CharLSTM",
                "corpus": corpus_name,
                "vocab_size": str(VOCAB_SIZE),
                "seq_len": str(SEQ_LEN_CHARS),
                "hidden": str(HIDDEN_CHARS),
                "layers": str(N_LAYERS),
                "epochs": str(CHAR_EPOCHS),
                "steps_per_epoch": str(STEPS_PER_EPOCH),
                "lr": str(CHAR_LR),
            }
        )
        for epoch in range(CHAR_EPOCHS):
            model.train()
            step_losses = []
            for _ in range(STEPS_PER_EPOCH):
                xb, yb = sample_batch(train_ids)
                logits = model(xb)
                # TODO: cross-entropy over ALL timesteps at once — flatten
                #   (batch, seq, vocab) -> (batch*seq, vocab) and
                #   (batch, seq) -> (batch*seq)
                loss = F.cross_entropy(____)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                step_losses.append(loss.item())

            train_ce = float(np.mean(step_losses))
            val_ce = mean_ce(val_ids, model)
            val_ppl = float(np.exp(val_ce))
            train_ces.append(train_ce)
            val_ces.append(val_ce)
            val_ppls.append(val_ppl)
            await run.log_metrics(
                {"train_ce": train_ce, "val_ce": val_ce, "val_perplexity": val_ppl},
                step=epoch + 1,
            )
            print(
                f"  [char_lstm] epoch {epoch+1}/{CHAR_EPOCHS}  "
                f"train CE={train_ce:.4f}  val CE={val_ce:.4f}  "
                f"val perplexity={val_ppl:.2f}"
            )
        await run.log_metrics(
            {"final_val_ce": val_ces[-1], "final_val_perplexity": val_ppls[-1]}
        )


await train_char_lstm()



### Checkpoint 2: the model beat the uniform baseline, honestly


In [ ]:
assert len(val_ces) == CHAR_EPOCHS
assert val_ces[-1] < val_ces[0] or train_ces[-1] < train_ces[0], (
    "training should reduce cross-entropy"
)
final_ppl = val_ppls[-1]
assert final_ppl < VOCAB_SIZE, (
    f"perplexity {final_ppl:.1f} must beat the uniform baseline {VOCAB_SIZE}"
)
assert final_ppl < 20.0, (
    f"perplexity {final_ppl:.1f} too high — a trained char-LSTM on this "
    "corpus reaches single digits; < 20 is the conservative floor"
)
print(
    f"\n  Final val CE={val_ces[-1]:.4f}, perplexity={final_ppl:.2f} "
    f"(uniform baseline: {VOCAB_SIZE})"
)
print("\n--- Checkpoint 2 passed --- perplexity beat the baseline\n")



## PHASE 4 — VISUALISE: perplexity curve + temperature samples


Autoregressive sampling: slide the window, sample, append.


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: perplexity and the temperature dial")
print("=" * 70)

fig_ppl, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(14, 5))
fig_ppl.suptitle("Character LSTM — cross-entropy and perplexity", fontsize=13)
epochs_x = range(1, CHAR_EPOCHS + 1)
ax_l.plot(epochs_x, train_ces, "o-", label="train CE", color="#2196F3")
ax_l.plot(epochs_x, val_ces, "s-", label="val CE", color="#FF5722")
ax_l.axhline(uniform_ce, color="gray", linestyle="--",
             label=f"uniform baseline ln({VOCAB_SIZE})={uniform_ce:.2f}")
ax_l.set_xlabel("Epoch")
ax_l.set_ylabel("Cross-entropy (nats)")
ax_l.legend(fontsize=9)
ax_l.grid(True, alpha=0.3)
ax_r.plot(epochs_x, val_ppls, "o-", color="#4CAF50", label="val perplexity")
ax_r.axhline(VOCAB_SIZE, color="gray", linestyle="--",
             label=f"uniform baseline = {VOCAB_SIZE}")
ax_r.set_xlabel("Epoch")
ax_r.set_ylabel("Perplexity = exp(CE)")
ax_r.legend(fontsize=9)
ax_r.grid(True, alpha=0.3)
fig_ppl.tight_layout()
fig_ppl.savefig(str(OUTPUT_DIR / "07_char_lstm_perplexity.png"), dpi=150)
plt.close(fig_ppl)
print(f"  Saved: {OUTPUT_DIR / '07_char_lstm_perplexity.png'}")


@torch.no_grad()
def generate(model: nn.Module, seed_text: str, n_chars: int = 300,
             temperature: float = 1.0) -> str:
    model.eval()
    ids = encode(seed_text)
    for _ in range(n_chars):
        window = torch.tensor(
            ids[-SEQ_LEN_CHARS:], dtype=torch.long, device=device
        ).unsqueeze(0)
        # TODO: last-timestep logits divided by the temperature, softmax,
        #   then SAMPLE (not argmax) one index with torch.multinomial
        logits = ____
        probs = F.softmax(logits, dim=-1)
        ids.append(____)
    return decode(ids)


SEED = "To be or not to be"
samples: dict[float, str] = {}
for temp in (0.5, 1.0, 1.5):
    torch.manual_seed(int(temp * 100))
    samples[temp] = generate(model, SEED, n_chars=300, temperature=temp)
    print(f"\n  ── temperature {temp} ──")
    print("  " + samples[temp].replace("\n", "\n  "))



### Checkpoint 3: artefacts and samples exist, temperatures differ


In [ ]:
import os

assert os.path.exists(OUTPUT_DIR / "07_char_lstm_perplexity.png")
for temp, s in samples.items():
    assert len(s) == len(SEED) + 300, f"t={temp}: wrong sample length"
    assert set(s) <= set(chars), f"t={temp}: sample left the vocabulary"
assert len({samples[0.5], samples[1.0], samples[1.5]}) == 3, (
    "temperatures should produce different samples"
)
print("\n--- Checkpoint 3 passed --- perplexity curve + samples verified\n")



## PHASE 5 — APPLY: drafting assistance at a professional services firm


DRAFTING ASSISTANT — MODEL CARD (measured, this run):

    Corpus:                 {corpus_name} ({len(text):,} chars)
    Vocabulary:             {VOCAB_SIZE} characters
    Uniform baseline:       perplexity {VOCAB_SIZE} (no learning)
    Trained model:          perplexity {final_ppl:.2f}
    Effective narrowing:    ~{compression:.0f}x fewer live candidates
                            per character than uniform guessing

  OPERATING GUIDANCE:
    Engagement letters / compliance boilerplate:  temperature 0.5
    Internal report drafting:                     temperature 1.0
    Brainstorming / naming workshops:             temperature 1.5

  STAKEHOLDER-READY OUTPUT:
    "The drafting model reads the partner's opening paragraph and
    proposes the next span of standard language. At each character it
    has narrowed the field from {VOCAB_SIZE} possibilities to about
    {final_ppl:.0f} — fluent enough for boilerplate. It learns FORM, not
    TRUTH: every proposal still gets a human read before it leaves the
    building. The creativity dial (temperature) is set per document
    class: 0.5 for engagement letters, 1.5 for brainstorming."


In [ ]:
# SCENARIO (anonymised, illustrative): a regional professional services
# firm drafts hundreds of engagement letters per quarter. Partners write
# the first paragraph; juniors assemble the boilerplate. The firm wants
# an autocomplete that proposes the next span of standard language.
#
# TWO QUESTIONS THE PARTNER WILL ASK — and the honest answers:
#
#   "How good is it?" — Perplexity translates directly:
#     our model's perplexity means that at each character it has
#     effectively narrowed ~70 possibilities down to ~perplexity-many.
#     That is a strong language model for boilerplate; it is NOT a
#     fact-checker. Character models learn FORM, not TRUTH.
#
#   "Can we tune how safe it plays?" — Yes: temperature.
#     T=0.5 for engagement letters (repetitive, safe, formulaic —
#     exactly what boilerplate wants); T=1.5 only for brainstorming
#     workshops where surprise is the point.
#
# WHAT WE WOULD NOT DO: ship raw character output into a client document
# without a human in the loop. The perplexity number tells you the model
# is fluent; it says nothing about whether the content is correct.

print("=" * 70)
print("  PHASE 5 — APPLY: drafting assistance, honestly scoped")
print("=" * 70)

compression = VOCAB_SIZE / final_ppl
print(
)

# ── Checkpoint 4: the perplexity interpretation is arithmetically sound
assert abs(np.log(final_ppl) - val_ces[-1]) < 1e-6, (
    "perplexity must equal exp(mean CE) by construction"
)
assert compression > 1.0, "trained model must narrow the candidate field"
print("--- Checkpoint 4 passed --- application quantified honestly\n")

await conn.close()



## REFLECTION


THEORY:
  [x] Generation = next-character classification, repeated; the
      vocabulary is the character set ({VOCAB_SIZE} symbols)
  [x] Perplexity = exp(mean CE): "as uncertain as a uniform choice over
      P characters". Uniform baseline = {VOCAB_SIZE}; ours = {final_ppl:.2f}
  [x] Temperature divides the logits: T<1 sharpens, T>1 flattens

  BUILD + TRAIN:
  [x] encode/decode with a sorted character vocabulary (round-trip
      verified on 1000 chars)
  [x] CharLSTM: embedding -> 2-layer LSTM -> per-timestep logits,
      {_n_params:,} parameters
  [x] {CHAR_EPOCHS} epochs x {STEPS_PER_EPOCH} random windows, gradient
      clipping, ExperimentTracker receipts (train CE {train_ces[0]:.3f}
      -> {train_ces[-1]:.3f})

  VISUALISE (the proof):
  [x] CE and perplexity curves against the uniform baseline line
  [x] Generated samples at three temperatures from the same seed
  [x] Vocabulary closure asserted: samples never leave the char set

  APPLY:
  [x] Drafting assistance with per-document-class temperature guidance
  [x] Perplexity translated for a partner: ~{compression:.0f}x narrower
      candidate field than uniform guessing
  [x] Stated limit: character models learn FORM, not TRUTH

  KEY INSIGHT: A language model's quality claim is only as good as its
  reference point. "Perplexity {final_ppl:.1f}" means nothing until you
  say "against a uniform baseline of {VOCAB_SIZE}". The same discipline
  you applied to accuracy baselines in M3 applies to generative text:
  measure, compare to the trivial baseline, and state what the metric
  does NOT cover.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

